# AU texts of four datasets
The work is done by the package `aposteriori`; this notebook calls it and shows the results. Each dataset is described in `aposteriori/datasets/<name>.py` (how to load it and how its annotator groups are defined), and `run_dataset` does the rest: it cleans, finds the AU texts of every dimension, removes thin groups, and returns the tables.

1. POPQUORN  2. Attitudes  3. Measuring Hate Speech  4. Kumar  5. Save

**AU rule** (`aposteriori.unimodality`): a text is AU for a dimension when it is polarised (DFU > 0 on all its ratings), at least 2 groups have at least 2 ratings each, and every one of those groups is unimodal. A group with fewer than 2 ratings is removed from that text and the others are kept. There is no minimum number of ratings per text.
**Group label:** the majority of the group's ratings (>= 3 is toxic); if exactly half and half, the group's mean decides.
**Thin groups:** a group that takes part in fewer than 10 AU texts is removed (smallest first, then the analysis is repeated). Its raters stay in the data and still count in the overall DFU of a text.

In [17]:
import importlib.util
import sys

if importlib.util.find_spec("aposteriori") is None:
    raise ModuleNotFoundError(
        "This notebook needs the package 'aposteriori'. Select the kernel 'aposteriori (package)', or install it in "
        "this Python: " + sys.executable + " -m pip install -e <path to aposteriori_package>")

import pandas as pd

from aposteriori import SPECS, find_root, run_dataset, save_au_tables

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)

ROOT = find_root()
results = {}

# 1. POPQUORN
1,500 texts, 262 raters, offensiveness 1-5, 5-10 ratings per text. Groups (`aposteriori/datasets/popquorn.py`):

| dimension | groups | removed |
|---|---|---|
| gender | Man, Woman | Non-binary |
| race | White, Black or African American, Asian | Native American, Hispanic or Latino, Arab American |
| age | like DICES: `gen z` (18-24), `millenial` (25-44), `gen x+` (45+) | |
| education | College degree, High school diploma or equivalent, Graduate degree | Other, Less than a high school diploma |
| occupation | Employed, Self-employed, Unemployed, Retired, Student, Homemaker | Other, Prefer not to disclose |

A rating with a removed value in any dimension is dropped.

In [18]:
results["popquorn"] = popquorn = run_dataset(SPECS["popquorn"], ROOT)
print(popquorn.describe())
popquorn.groups

ratings 13,036 -> 11,352 | items 1,500 -> 1,500
ratings with a removed value, per dimension: {'gender': 350, 'race': 598, 'age': 0, 'education': 535, 'occupation': 300}
ratings left per item: {3: 2, 4: 15, 5: 60, 6: 202, 7: 387, 8: 504, 9: 267, 10: 63}


,,ratings
Asian,race,899
Black or African American,race,1535
College degree,education,5537
Employed,occupation,5826
Graduate degree,education,2381
High school diploma or equivalent,education,3434
Homemaker,occupation,494
Man,gender,5526
Retired,occupation,1346
Self-employed,occupation,1593


## AU texts: every group, and the thin ones removed

In [19]:
popquorn.starting_list

AU texts at start  AU texts at the end  removed
dimension  group                                                                             
gender     Man                                               14                 14.0    False
           Woman                                             14                 14.0    False
race       Asian                                              8                  NaN     True
           Black or African American                         12                 13.0    False
           White                                             20                 13.0    False
age        gen x+                                            39                 55.0    False
           gen z                                              7                  NaN     True
           millenial                                         36                 55.0    False
education  College degree                                    44                 44.0    False
           Graduate degree                                   17                 17.0    False
           High school diploma or equivalent                 38                 38.0    False
occupation Employed                                          54                 52.0    False
           Homemaker                                          3                  NaN     True
           Retired                                           11                 11.0    False
           Self-employed                                     27                 28.0    False
           Student                                           11                 11.0    False
           Unemployed                                        10                 10.0    False

## Result

In [20]:
print(f"{len(popquorn.au)} rows ({popquorn.au['item_id'].nunique()} different texts)")
display(popquorn.per_dimension)
display(popquorn.per_group)
display(popquorn.removal_cost)

183 rows (154 different texts)


,texts,survived (AU),not survived,groups kept,groups removed
gender,1500,14,1486,2,-
race,1500,13,1487,2,Asian
age,1500,55,1445,2,gen z
education,1500,47,1453,3,-
occupation,1500,54,1446,5,Homemaker


AU texts  toxic  civil
dimension  group                                                    
gender     Man                                      14      5      9
           Woman                                    14      6      8
race       Black or African American                13     11      2
           White                                    13      1     12
age        gen x+                                   55     17     38
           millenial                                55      7     48
education  College degree                           44     10     34
           Graduate degree                          17     10      7
           High school diploma or equivalent        38      8     30
occupation Employed                                 52     15     37
           Retired                                  11      3      8
           Self-employed                            28      2     26
           Student                                  11      3      8
           Unemployed                               10      1      9

,,ratings,AU texts it took part in,AU texts at start,AU texts at end,lost,gained
dimension,group removed,,,,,,
race,Asian,899,8,20,13,8,1
age,gen z,1345,7,40,55,5,20
occupation,Homemaker,494,3,56,54,3,1


# 2. Attitudes
626 tweets, 3,463 ratings (`toyou`, 1-5), 3 to 12 ratings per text; the annotator attributes are lists parallel to the ratings, and there is no annotator id (`aposteriori/datasets/attitudes.py`).

| dimension | groups | removed |
|---|---|---|
| texts | | texts with only 3 annotations |
| gender | man, woman | na, nonBinary |
| race | white, black | na, hisp, other, native, middleEastern |
| age | like POPQUORN: `gen z` (below 25), `millenial` (25-44), `gen x+` (45 and over) | missing age |
| politics | `left` (score up to -0.25), `center` (above -0.25 up to 0.25), `right` (above 0.25) | |
| traditionalism | `not_high` (score 1-3), `high` (score 4-5) | |

In [21]:
results["attitudes"] = attitudes = run_dataset(SPECS["attitudes"], ROOT)
print(attitudes.describe())
attitudes.groups

ratings 3,463 -> 3,354 | items 626 -> 625
prefilter removed 3 ratings (before the group cleaning)
ratings with a removed value, per dimension: {'gender': 65, 'race': 80, 'age': 41, 'politics': 0, 'traditionalism': 0}
ratings left per item: {2: 1, 3: 40, 4: 83, 5: 111, 6: 389, 11: 1}


,,ratings
black,race,1131
center,politics,460
gen x+,age,809
gen z,age,4
high,traditionalism,2088
left,politics,1741
man,gender,1893
millenial,age,2541
not_high,traditionalism,1266
right,politics,1153


## AU texts: every group, and the thin ones removed

In [22]:
attitudes.starting_list

AU texts at start  AU texts at the end  removed
dimension      group                                                     
gender         man                       10                 10.0    False
               woman                     10                 10.0    False
race           black                     14                 14.0    False
               white                     14                 14.0    False
age            gen x+                     6                  6.0    False
               gen z                      0                  0.0    False
               millenial                  6                  6.0    False
politics       center                     9                  NaN     True
               left                      17                 12.0    False
               right                     11                 12.0    False
traditionalism high                       9                  9.0    False
               not_high                   9                  9.0    False

## Result

In [23]:
print(f"{len(attitudes.au)} rows ({attitudes.au['item_id'].nunique()} different texts)")
display(attitudes.per_dimension)
display(attitudes.per_group)
display(attitudes.removal_cost)

51 rows (41 different texts)


,texts,survived (AU),not survived,groups kept,groups removed
gender,625,10,615,2,-
race,625,14,611,2,-
age,625,6,619,3,-
politics,625,12,613,2,center
traditionalism,625,9,616,2,-


AU texts  toxic  civil
dimension      group                            
gender         man              10      5      5
               woman            10      5      5
race           black            14     12      2
               white            14      5      9
age            gen x+            6      2      4
               gen z             0      0      0
               millenial         6      4      2
politics       left             12      8      4
               right            12      8      4
traditionalism high              9      6      3
               not_high          9      5      4

,,ratings,AU texts it took part in,AU texts at start,AU texts at end,lost,gained
dimension,group removed,,,,,,
politics,center,460,9,17,12,6,1


# 3. Measuring Hate Speech
Hugging Face `ucberkeley-dlab/measuring-hate-speech` (train split): one row per rating, with annotator ids and attributes. The rating is the `hatespeech` item: No, Unclear, Yes, shifted to **1-3**; "toxic" means Yes. Only comments with at least 4 ratings are kept, since the others cannot reach 2 + 2 (`aposteriori/datasets/mhs.py`; the file is read from `datasets/mhs/train.parquet` if it exists, otherwise downloaded).

| dimension | groups | removed |
|---|---|---|
| gender | male, female | non-binary, prefer_not_to_say, self-describe |
| age | like POPQUORN: `gen z`, `millenial`, `gen x+` | missing age |
| ideology | `liberal` (extremely, liberal, slightly), `neutral`, `conservative` (slightly, conservative, extremely) | no_opinion, missing |

Race is not used.

In [ ]:
results["mhs"] = mhs = run_dataset(SPECS["mhs"], ROOT)
print(mhs.describe())
mhs.groups

## AU texts: every group, and the thin ones removed

In [ ]:
mhs.starting_list

AU texts at start  AU texts at the end  removed
dimension group                                                        
gender    female                       43                 43.0    False
          male                         43                 43.0    False
age       gen x+                       31                 30.0    False
          gen z                         5                  NaN     True
          millenial                    34                 30.0    False
ideology  conservative                 22                 22.0    False
          liberal                      29                 29.0    False
          neutral                      13                 13.0    False

## Result

In [ ]:
print(f"{len(mhs.au)} rows ({mhs.au['item_id'].nunique()} different texts)")
display(mhs.per_dimension)
display(mhs.per_group)
display(mhs.removal_cost)

105 rows (95 different texts)


,texts,survived (AU),not survived,groups kept,groups removed
gender,5990,43,5947,2,-
age,5990,30,5960,2,gen z
ideology,5990,32,5958,3,-


AU texts  toxic  civil
dimension group                               
gender    female              43     30     13
          male                43     13     30
age       gen x+              30     22      8
          millenial           30      8     22
ideology  conservative        22     15      7
          liberal             29     12     17
          neutral             13      3     10

,,ratings,AU texts it took part in,AU texts at start,AU texts at end,lost,gained
dimension,group removed,,,,,,
age,gen z,5544,5,35,30,5,0


# 4. Kumar
107,620 comments, exactly 5 ratings each, scores 0-4 shifted to **1-5**; every rating carries the rater's attributes and there is no rater id (`aposteriori/datasets/kumar.py`). Reading the 450 MB file takes about 20 seconds.

| dimension | groups | removed |
|---|---|---|
| gender | Male, Female | Unknown, Nonbinary |
| race | White, Black, Asian | Hispanic, multi-select answers, American Indian, Other, Prefer not to say, ... |
| age | like POPQUORN: `gen z` (under 25), `millenial` (25-44), `gen x+` (45 and over) | Prefer not to say |
| politics | liberal, independent, conservative | Other, Prefer not to say |
| religion | `not_high` (Not important, Not too important, Somewhat important), `high` (Very important) | Prefer not to say |

In [ ]:
results["kumar"] = kumar = run_dataset(SPECS["kumar"], ROOT)
print(kumar.describe())
kumar.groups

ratings 538,100 -> 450,160 | items 107,620 -> 107,620
ratings with a removed value, per dimension: {'gender': 6860, 'race': 56120, 'age': 1140, 'politics': 34000, 'religion': 7400}
ratings left per item: {1: 300, 2: 3640, 3: 16720, 4: 42380, 5: 44580}


,,ratings
Asian,race,28860
Black,race,61720
Female,gender,234940
Male,gender,215220
White,race,359580
conservative,politics,132240
gen x+,age,111280
gen z,age,46580
high,religion,146040
independent,politics,125120


## AU texts: every group, and the thin ones removed

In [ ]:
kumar.starting_list

AU texts at start  AU texts at the end  removed
dimension group                                                        
gender    Female                     1224                 1224    False
          Male                       1224                 1224    False
race      Asian                        83                   83    False
          Black                       592                  592    False
          White                       659                  659    False
age       gen x+                      880                  880    False
          gen z                       243                  243    False
          millenial                   953                  953    False
politics  conservative               1228                 1228    False
          independent                1143                 1143    False
          liberal                    1477                 1477    False
religion  high                       1237                 1237    False
          not_high                   1237                 1237    False

## Result

In [ ]:
print(f"{len(kumar.au)} rows ({kumar.au['item_id'].nunique()} different texts)")
display(kumar.per_dimension)
display(kumar.per_group)
display(kumar.removal_cost)

6090 rows (5209 different texts)


,texts,survived (AU),not survived,groups kept,groups removed
gender,107620,1224,106396,2,-
race,107620,667,106953,3,-
age,107620,1038,106582,3,-
politics,107620,1924,105696,3,-
religion,107620,1237,106383,2,-


AU texts  toxic  civil
dimension group                               
gender    Female            1224    501    723
          Male              1224    768    456
race      Asian               83     38     45
          Black              592    511     81
          White              659    110    549
age       gen x+             880    384    496
          gen z              243     71    172
          millenial          953    403    550
politics  conservative      1228    674    554
          independent       1143    296    847
          liberal           1477    434   1043
religion  high              1237   1070    167
          not_high          1237    207   1030

""


# 5. Save
Files in `new_results/` (written by `aposteriori.export.save_au_tables`):

| file | content |
|---|---|
| `popquorn/popquorn_au.csv`, `attitudes/attitudes_au.csv`, `mhs/mhs_au.csv`, `kumar/kumar_au.csv` | the AU texts of one dataset |
| `aposteriori_all.csv` | all four datasets together |
| `aposteriori_for_models.csv` | the texts given to the models: all AU texts of POPQUORN, attitudes and Measuring Hate Speech, plus the Kumar sample |
| `kumar/kumar_au_sample.csv` | 20 random AU texts per dimension of Kumar (fixed seed 42) |

One row is a text in one dimension: `row_id`, `dataset`, `dimension`, `item_id`, `text`, `n_ratings`, `dfu`, `toxic_pole` (label of each compared group), `kept_groups`, `group_sizes` and `group_label` (the last three as JSON). A text that is AU in several dimensions has one row for each.

In [ ]:
saved = save_au_tables({name: result.au for name, result in results.items()}, ROOT / "new_results",
                       exclude_for_models=("kumar",),
                       sample={"kumar": {"n": 20, "seed": 42, "dimensions": list(SPECS["kumar"].dimensions)}})
saved

,rows,different texts,MB
file,,,
popquorn/popquorn_au.csv,183,154,0.08
attitudes/attitudes_au.csv,51,41,0.02
mhs/mhs_au.csv,105,95,0.03
kumar/kumar_au.csv,6090,5209,2.05
aposteriori_all.csv,6429,5499,2.18
aposteriori_for_models.csv,339,290,0.13
kumar/kumar_au_sample.csv,100,100,0.03


In [ ]:
df_temp=pd.read_csv("../../new_results/aposteriori_for_models.csv")
df_temp.groupby("dataset")["dimension"].value_counts()

dataset    dimension     
attitudes  race              14
           politics          12
           gender            10
           traditionalism     9
           age                6
mhs        gender            43
           ideology          32
           age               30
popquorn   age               55
           occupation        54
           education         47
           gender            14
           race              13
Name: count, dtype: int64